# Inspeção manual da amostra (plano §3.2, item 4)

Casos ambíguos e suspeitos são resolvidos por **inspeção manual documentada**.
Este notebook lista os 100 selecionados, sinaliza suspeitos de não-software e
organiza os ambíguos para revisão.

**Fluxo de decisão** (auditável):
1. Inspecione os sinalizados abaixo (a coluna `url` abre o repositório).
2. Para excluir um repo: adicione-o a `config/sampling.yaml` →
   `exclusions.repos` com um comentário do motivo.
3. Re-execute `govscore sample` (rápido — buscas cacheadas) e depois
   `govscore run` (retomável — só os substitutos são extraídos).
4. `sensitivity`, `validate` e `make figures` regeneram o restante.

Como abrir: `make lab` (ou `uv run jupyter lab`) na raiz do repositório.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
import yaml

ROOT = Path.cwd()
while not (ROOT / "config" / "metrics.yaml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.max_colwidth", 100)

d = yaml.safe_load((ROOT / "config" / "sample_full.yaml").read_text())
full = pd.DataFrame(d["full"])
amb = pd.DataFrame(d["ambiguous"])
full["url"] = "https://github.com/" + full.repo
print(f"selecionados: {len(full)} | ambíguos: {len(amb)} | gerado em {d['generated_at']}")
full.groupby("archetype").size()

selecionados: 100 | ambíguos: 406 | gerado em 2026-07-20


archetype
club          25
federation    25
stadium       25
toy           25
dtype: int64

## Suspeitos para inspeção

Duas heurísticas: padrão de nome típico de não-software (a triagem por
tópicos não pega tudo) e ausência de resposta humana nas issues amostradas
(sinal fraco de comunidade — vindo do QA da extração).

In [2]:
SUSPEITOS = (r"leetcode|interview|awesome|explore|roadmap|tutorial|study|"
             r"notes|book|course|cheat|free[-_]?vpn|subscription")
qa = pd.json_normalize(json.loads(
    (ROOT / "data/processed/full_metrics.json").read_text())["results"])
sem_resposta = set(qa.loc[qa["responsiveness.n_first_responses"] == 0, "repo"])

full["flag_nome"] = full.repo.str.lower().str.contains(SUSPEITOS, regex=True)
full["flag_sem_resposta"] = full.repo.isin(sem_resposta)
suspeitos = full[full.flag_nome | full.flag_sem_resposta]
suspeitos[["repo", "archetype", "language", "stars",
           "active_contributors_2plus", "flag_nome", "flag_sem_resposta", "url"]]

,repo,archetype,language,stars,active_contributors_2plus,flag_nome,flag_sem_resposta,url
0,torvalds/linux,federation,c,239888,101,False,True,https://github.com/torvalds/linux
12,FFmpeg/FFmpeg,federation,c,62223,100,False,True,https://github.com/FFmpeg/FFmpeg
16,git/git,federation,c,62105,100,False,True,https://github.com/git/git
36,MisterBooo/LeetCodeAnimation,stadium,java,76631,1,True,False,https://github.com/MisterBooo/LeetCodeAnimation
48,gitlabhq/gitlabhq,stadium,ruby,24501,2,False,True,https://github.com/gitlabhq/gitlabhq
55,github/explore,club,ruby,4819,72,True,True,https://github.com/github/explore
83,fustyles/Arduino,toy,c++,426,1,False,True,https://github.com/fustyles/Arduino
87,AITabby/opencodex,toy,typescript,377,3,False,True,https://github.com/AITabby/opencodex
93,yolfinance/yolfi-agent,toy,javascript,212,2,False,True,https://github.com/yolfinance/yolfi-agent
94,EFanZh/LeetCode,toy,rust,228,1,True,True,https://github.com/EFanZh/LeetCode


## Casos ambíguos registrados

Por motivo — os "fora das faixas" são as zonas deliberadas da matriz §3.1
(não são erro); truncados e inativos merecem olhada se algum substituto for
necessário.

In [3]:
amb.reason.str.slice(0, 40).value_counts()

reason
fora das faixas da §3.1                     354
contagem truncada em 3000 commits — conf     40
sem commits no branch default na janela      12
Name: count, dtype: int64

In [4]:
# ambíguos com contagem truncada (piso de atividade alto) — candidatos a
# inspeção caso um estrato precise de substituto
amb[amb.reason.str.startswith("contagem")].assign(
    url=lambda x: "https://github.com/" + x.repo
).sort_values("stars", ascending=False).head(20)

,repo,language,stars,active_contributors_2plus,classified_at,reason,url
6,openclaw/openclaw,typescript,383552,94,2026-07-20,contagem truncada em 3000 commits — confirmar manualmente,https://github.com/openclaw/openclaw
13,n8n-io/n8n,typescript,197153,88,2026-07-20,contagem truncada em 3000 commits — confirmar manualmente,https://github.com/n8n-io/n8n
18,microsoft/vscode,typescript,187717,73,2026-07-20,contagem truncada em 3000 commits — confirmar manualmente,https://github.com/microsoft/vscode
25,anomalyco/opencode,typescript,187713,32,2026-07-20,contagem truncada em 3000 commits — confirmar manualmente,https://github.com/anomalyco/opencode
79,open-webui/open-webui,python,146049,35,2026-07-20,contagem truncada em 3000 commits — confirmar manualmente,https://github.com/open-webui/open-webui
21,vercel/next.js,javascript,141024,48,2026-07-20,contagem truncada em 3000 commits — confirmar manualmente,https://github.com/vercel/next.js
24,denoland/deno,rust,107748,66,2026-07-20,contagem truncada em 3000 commits — confirmar manualmente,https://github.com/denoland/deno
34,oven-sh/bun,rust,94895,28,2026-07-20,contagem truncada em 3000 commits — confirmar manualmente,https://github.com/oven-sh/bun
30,bitcoin/bitcoin,c++,89654,71,2026-07-20,contagem truncada em 3000 commits — confirmar manualmente,https://github.com/bitcoin/bitcoin
15,spring-projects/spring-boot,java,81128,35,2026-07-20,contagem truncada em 3000 commits — confirmar manualmente,https://github.com/spring-projects/spring-boot


## Registro de decisões

| data | repo | decisão | motivo |
|---|---|---|---|
| _preencher_ | | mantido / excluído | |

Decisões efetivadas entram em `config/sampling.yaml` (`exclusions.repos`)
e, se mudarem o catálogo ou critérios, em `docs/decisions/`.